# Critical Input DEQN: Repair-Aware Taylor Rule

This notebook trains the adaptation-aware Taylor rule, `policy="repair_aware"`. The rule starts from the standard inflation Taylor rate and lowers it only when the zero-rent bottleneck pressure is near binding and the repair value is close to the private investment threshold.

In [ ]:
# Configure paths and repair-aware Taylor settings.
from pathlib import Path
import json
import subprocess
import sys
import torch

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent

ARTIFACT_ROOT = ROOT / 'baseline_artifacts' / 'critical_input_deqn'

def first_existing(paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    raise FileNotFoundError('No checkpoint found:\n' + '\n'.join(str(p) for p in paths))

NATURAL_CKPT = first_existing([
    ARTIFACT_ROOT / 'natural' / 'checkpoints' / 'natural_best.pt',
    ARTIFACT_ROOT / 'natural' / 'natural.pt',
])
OUT = ARTIFACT_ROOT / 'repair_aware_taylor'
OUT.mkdir(parents=True, exist_ok=True)

RULE_STEPS = 8_000
QMC_TRAIN = 256
QMC_VAL = 512
N_VAL_STATES = 1024
HIDDEN_WIDTH = 192
HIDDEN_DEPTH = 2
LOG_EVERY = 100
BATCH_SIZE = 2048
SIM_BATCH_SIZE = 512
EPISODE_LENGTH = 20
EPISODE_UPDATES_PER_EPISODE = 2
EPISODE_BROAD_SHARE = 0.50
CHECKPOINT_EVERY = 1000
STOP_VAL_STATES = 512
SCENARIO_Q_WEIGHT = 25.0
CALM_ANCHOR_WEIGHT = 5.0
CALM_RESIDUAL_WEIGHT = 5.0
SCENARIO_BURNIN = 5
SCENARIO_HORIZON = 10
SCENARIO_LOSS_INTERVAL = 25
TARGET_SCENARIO_Q_RMS = 1e-2
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = 'float64'

print('ROOT:', ROOT)
print('natural:', NATURAL_CKPT)
print('output:', OUT)
print('device:', DEVICE)

def run_stream(cmd, cwd=ROOT):
    print('Running:', ' '.join(map(str, cmd)))
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    code = proc.wait()
    if code != 0:
        raise subprocess.CalledProcessError(code, cmd)


In [ ]:
# Train the repair-aware Taylor network.
cmd = [
    sys.executable, '-u', '-m', 'src.critical_input_deqn.run_train',
    '--output-dir', str(OUT),
    '--policies', 'repair_aware',
    '--natural-checkpoint', str(NATURAL_CKPT),
    '--rule-steps', str(RULE_STEPS),
    '--rule-trainer', 'episode',
    '--qmc-train', str(QMC_TRAIN),
    '--qmc-val', str(QMC_VAL),
    '--n-val-states', str(N_VAL_STATES),
    '--hidden-width', str(HIDDEN_WIDTH),
    '--hidden-depth', str(HIDDEN_DEPTH),
    '--device', DEVICE,
    '--dtype', DTYPE,
    '--stop-val-states', str(STOP_VAL_STATES),
    '--log-every', str(LOG_EVERY),
    '--batch-size', str(BATCH_SIZE),
    '--sim-batch-size', str(SIM_BATCH_SIZE),
    '--episode-length', str(EPISODE_LENGTH),
    '--episode-updates-per-episode', str(EPISODE_UPDATES_PER_EPISODE),
    '--episode-broad-share', str(EPISODE_BROAD_SHARE),
    '--checkpoint-every', str(CHECKPOINT_EVERY),
    '--rule-scenario-q-weight', str(SCENARIO_Q_WEIGHT),
    '--rule-calm-anchor-weight', str(CALM_ANCHOR_WEIGHT),
    '--rule-calm-residual-weight', str(CALM_RESIDUAL_WEIGHT),
    '--rule-scenario-burnin', str(SCENARIO_BURNIN),
    '--rule-scenario-horizon', str(SCENARIO_HORIZON),
    '--rule-scenario-loss-interval', str(SCENARIO_LOSS_INTERVAL),
    '--target-scenario-q-rms', str(TARGET_SCENARIO_Q_RMS),
]
run_stream(cmd)


In [ ]:
# Inspect out-of-sample residual diagnostics for repair-aware Taylor.
eval_path = OUT / 'repair_aware_eval.json'
if not eval_path.exists():
    raise FileNotFoundError(f'Missing eval file. If training was interrupted, use {OUT / "checkpoints" / "repair_aware_best.pt"} in a diagnostics cell.')
with eval_path.open('r', encoding='utf-8') as fh:
    repair_aware_eval = json.load(fh)
repair_aware_eval


In [ ]:
# Optional: compare rule-policy validation diagnostics if the other Taylor rules already exist.
import pandas as pd

paths = {
    'fixed': ARTIFACT_ROOT / 'fixed_taylor' / 'fixed_eval.json',
    'natural_rate_adjusted': ARTIFACT_ROOT / 'modified_taylor' / 'ba_eval.json',
    'bottleneck': ARTIFACT_ROOT / 'bottleneck_taylor' / 'bottleneck_eval.json',
    'repair_aware': OUT / 'repair_aware_eval.json',
}
keys = [
    'rms', 'max_abs', 'hh_euler.rms', 'resource.rms', 'price_index.rms',
    'calvo_S.rms', 'calvo_F.rms', 'Q.rms', 'scenario_Q.rms',
    'scenario_Q.D_3x.event', 'calm_anchor.rms', 'calm_residual.rms',
    'exact_cap_product_scaled.rms', 'exact_repair_projection.rms',
]
rows = []
for name, path in paths.items():
    if not path.exists():
        continue
    with path.open('r', encoding='utf-8') as fh:
        data = json.load(fh)
    rows.append({'policy': name, **{key: data.get(key) for key in keys}})
pd.DataFrame(rows).set_index('policy')
